# TracIn - NLP Text Classification (DBPedia)

This notebook implements the TracIn algorithm for a text classification task on the DBPedia dataset, as described in **Section 5.2** of the paper _Estimating Training Data Influence by Tracing Gradient Descent_.

**CRITICAL UPDATE:** This version **completely removes `torchtext`** (which causes `libtorchtext.so` OSError on newer PyTorch versions). Instead, we use the `datasets` library from Hugging Face and build a vocabulary from scratch using pure Python.

In [17]:
!pip install datasets tqdm

In [18]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import numpy as np
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)

Using device: cuda


### Application: Text Classification (DBPedia)
**Reference: Paper Section 5.2**

We apply `TracIn` on the DBPedia ontology dataset. The task is to predict the ontology class using the title and abstract from Wikipedia.
We train a Simple Word-Embedding Model (SWEM) and apply `TracInCP` using evenly spaced checkpoints. The gradients are taken with respect to the last fully connected layer.


## 1. Load Data (Using HuggingFace `datasets`)
Instead of `torchtext`, we download the DBPedia14 dataset directly using the Hugging Face `datasets` library.

In [19]:
print("Downloading DBPedia 14...")
dataset = load_dataset("dbpedia_14")

train_data = dataset["train"]
test_data = dataset["test"]

# Subset for faster training demonstration ( 20k out of 560k)
SUBSET_SIZE = 20000
train_data = train_data.select(range(SUBSET_SIZE))

print(f"Training examples: {len(train_data)}")

README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Training examples: 20000


## 2. Tokenization and Vocabulary (Pure Python)
We split strings by whitespace and keep the top 40,000 most frequent words.

In [20]:
def basic_tokenizer(text):
    # Simple lowercase and split
    return str(text).lower().split()

counter = Counter()
for example in tqdm(train_data, desc="Building vocabulary"):
    counter.update(basic_tokenizer(example["content"]))

# Keep top 40,000 words
VOCAB_SIZE = 40000
vocab = {"<pad>": 0, "<unk>": 1}

for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Final Vocabulary Size: {len(vocab)}")

def encode_text(text):
    return [vocab.get(w, vocab["<unk>"]) for w in basic_tokenizer(text)]

Building vocabulary:   0%|          | 0/20000 [00:00<?, ?it/s]

Final Vocabulary Size: 40000


In [21]:
class DBPediaDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        # Label in HF dataset is 0-13
        return item["label"], item["content"]

train_pytorch_dataset = DBPediaDataset(train_data)
test_pytorch_dataset = DBPediaDataset(test_data)

def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    for label, text in batch:
        label_list.append(label)
        encoded = encode_text(text)
        if len(encoded) == 0: encoded = [0] # Handle empty strings
        processed_text = torch.tensor(encoded, dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))
        
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)

train_dataloader = DataLoader(train_pytorch_dataset, batch_size=64, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(test_pytorch_dataset, batch_size=64, shuffle=False, collate_fn=collate_batch)

## 3. Simple Word-Embedding Model (SWEM)

In [22]:
class SWEM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(SWEM, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False, mode="mean")
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

embed_dim = 64
num_class = 14 # DBPedia has 14 classes
model = SWEM(len(vocab), embed_dim, num_class).to(device)

## 4. Training with Checkpoints

In [23]:
epochs = 6
lr = 1.0
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

checkpoints = []
checkpoint_epochs = [2, 4, 6] # Save checkpoints periodically

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_acc, total_count = 0, 0, 0
    
    for label, text, offsets in tqdm(train_dataloader, desc=f"Epoch {epoch}", leave=False):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        
    print(f"Epoch {epoch} | Loss: {total_loss/len(train_dataloader):.4f} | Acc: {total_acc/total_count:.4f}")
    
    if epoch in checkpoint_epochs:
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "learning_rate": lr
        }
        checkpoints.append(checkpoint)
        print(f"--- Saved checkpoint at epoch {epoch} ---")

Epoch 1:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.1548 | Acc: 0.9911


Epoch 2:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.0018 | Acc: 1.0000
--- Saved checkpoint at epoch 2 ---


Epoch 3:   0%|          | 0/313 [00:00<?, ?it/s]

...

Epoch 4 | Loss: 0.0007 | Acc: 1.0000
--- Saved checkpoint at epoch 4 ---


Epoch 5:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 5 | Loss: 0.0005 | Acc: 1.0000


Epoch 6:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 6 | Loss: 0.0004 | Acc: 1.0000
--- Saved checkpoint at epoch 6 ---


## 5. Efficient TracIn for NLP

### Understanding TracIn & Checkpoints Heuristic
**Reference: Paper Section 3.1, 3.2 & 3.3**

TracIn decomposes the difference between the loss of the test point at the end of training versus at the beginning of training.
The change in loss at a single step $t$ is approximated as the dot product of the test loss gradient and the parameter update vector.
$$TracIn(z, z') = \sum_{t: z_t=z} \eta_t \nabla\ell(w_t, z') \cdot \nabla\ell(w_t, z)$$

Since tracing parameters at every iteration is impractical, `TracInCP` uses checkpoints stored at regular intervals:
$$TracInCP(z, z') = \sum_{i=1}^k \eta_i \nabla\ell(w_{t_i}, z) \cdot \nabla\ell(w_{t_i}, z')$$



In [24]:
def compute_last_layer_gradient_nlp(model, text, offsets, label, criterion, device):
    model.zero_grad()
    output = model(text, offsets)
    loss = criterion(output, label)
    loss.backward()
    
    # SWEM last layer is named 'fc'
    grad_weight = model.fc.weight.grad.view(-1)
    grad_bias = model.fc.bias.grad.view(-1)
    grad = torch.cat([grad_weight, grad_bias])
    
    return grad.detach().cpu()

### Proponents and Opponents
**Reference: Paper Remark 3.2**

*   **Proponents**: Training examples with a positive influence score, indicating they reduce loss.
*   **Opponents**: Training examples with a negative influence score, indicating they increase loss.


In [25]:
def find_proponents_opponents_nlp(test_example, train_dataset, checkpoints, criterion, device, top_k=3, max_train=5000):
    test_label, test_text_str = test_example
    test_label_tensor, test_text_tensor, test_offsets = collate_batch([(test_label, test_text_str)])
    
    num_examples = min(max_train, len(train_dataset))
    scores = np.zeros(num_examples)
    
    for checkpoint in checkpoints:
        model = SWEM(len(vocab), embed_dim, num_class).to(device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()
        lr = checkpoint["learning_rate"]
        
        # Compute test gradient ONCE per checkpoint
        grad_test = compute_last_layer_gradient_nlp(
            model, test_text_tensor, test_offsets, test_label_tensor, criterion, device
        )
        
        # Loop over training dataset
        for idx in tqdm(range(num_examples), desc=f"Tracing Train Data (Epoch {checkpoint['epoch']})"):
            train_label, train_text_str = train_dataset[idx]
            train_label_tensor, train_text_tensor, train_offsets = collate_batch([(train_label, train_text_str)])
            
            grad_train = compute_last_layer_gradient_nlp(
                model, train_text_tensor, train_offsets, train_label_tensor, criterion, device
            )
            
            dot_product = torch.dot(grad_train, grad_test).item()
            scores[idx] += lr * dot_product
            
    indexed_scores = [(scores[idx], idx) for idx in range(num_examples)]
    indexed_scores.sort(reverse=True)
    return indexed_scores[:top_k], indexed_scores[-top_k:]

In [26]:
# Select a test example
test_idx = 10 # Arbitrary test example
test_example = test_pytorch_dataset[test_idx]
test_label_id, test_text = test_example

# Mapping for DBPedia labels
label_names = [
    "Company", "EducationalInstitution", "Artist", "Athlete", "OfficeHolder",
    "MeanOfTransportation", "Building", "NaturalPlace", "Village", "Animal",
    "Plant", "Album", "Film", "WrittenWork"
]

print(f"--- TEST EXAMPLE ---")
print(f"Class: {label_names[test_label_id]} ({test_label_id})")
print(f"Text: {test_text[:300]}...\n")

# Find proponents and opponents
# We search through the first 2000 training examples for speed
proponents, opponents = find_proponents_opponents_nlp(
    test_example, train_pytorch_dataset, checkpoints, criterion, device, top_k=3, max_train=2000
)

print("\n=== TOP 3 PROPONENTS (Helpful Examples) ===")
for score, idx in proponents:
    label, text = train_pytorch_dataset[idx]
    print(f"Score: {score:.4f} | Class: {label_names[label]} ({label})")
    print(f"Text: {text[:200]}...\n")

print("\n=== TOP 3 OPPONENTS (Harmful Examples) ===")
for score, idx in opponents[::-1]: # Reverse to show most harmful first
    label, text = train_pytorch_dataset[idx]
    print(f"Score: {score:.4f} | Class: {label_names[label]} ({label})")
    print(f"Text: {text[:200]}...\n")

--- TEST EXAMPLE ---
Class: Company (0)
Text:  MEPC plc is a leading British-based property investment and development business. It is headquartered in London. It used to be listed on the London Stock Exchange and was once a constituent of the FTSE 100 Index. It is however now owned by the British Telecom Pension Fund and the Royal Mail Pension...



Tracing Train Data (Epoch 2):   0%|          | 0/2000 [00:00<?, ?it/s]

Tracing Train Data (Epoch 4):   0%|          | 0/2000 [00:00<?, ?it/s]

Tracing Train Data (Epoch 6):   0%|          | 0/2000 [00:00<?, ?it/s]


=== TOP 3 PROPONENTS (Helpful Examples) ===
Score: 0.0000 | Class: Company (0)
Text:  IBISWorld is an Australian research company....

Score: 0.0000 | Class: Company (0)
Text:  Post Bank of Iran (Persian: پست بانک ایران‎) is an Iranian bank....

Score: 0.0000 | Class: Company (0)
Text:  Dialog Axiata PLC DBA Dialog (formerly known as Dialog Telekom) is Sri Lanka's largest telecommunications service provider with the country's largest mobile phone network of over 8.7 million subscrib...


=== TOP 3 OPPONENTS (Harmful Examples) ===
Score: 0.0000 | Class: Company (0)
Text:  Dockwise Ltd. is a Bermuda-based holding company in the marine transport industry. The Steven Mitchell Investment Bank (NYSE:UTSM) is handling the IPO....

Score: 0.0000 | Class: Company (0)
Text:  Julius Blum GmbH (commonly referred to as Blum) is an international company that produces hinge- lift- and runner-systems and the appropriate assembly tools for the cabinet making and furniture indus...

Score: 0.0000 | Cla